# 02 — NDVI Exploratory Data Analysis

This notebook explores the Sentinel-2 NDVI data stored in S3 under `standardized/ndvi/`.

**Data source:** COPERNICUS/S2_SR_HARMONIZED, monthly median composites, cloud-masked.

**Key questions:**
- How many mandis have NDVI data? Coverage by state?
- What is the temporal distribution (monthly observations per mandi)?
- What is the NDVI value distribution? Any outliers?
- How does NDVI vary across states and seasons?
- Forward-fill to daily: does it look reasonable?

In [ ]:
import io
import re
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True

## 1. Load NDVI from S3

In [ ]:
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name != "notebooks" else Path(".").resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "aws_config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

session = boto3.Session(
    profile_name=cfg["aws"].get("profile"),
    region_name=cfg["aws"].get("region", "ap-south-1"),
)
s3 = session.client("s3")
bucket = cfg["s3"]["bucket"]
print(f"Bucket: {bucket}")

In [ ]:
# Read all NDVI parquet files from S3
prefix = "standardized/ndvi/"
keys, token = [], None
while True:
    kwargs = dict(Bucket=bucket, Prefix=prefix)
    if token:
        kwargs["ContinuationToken"] = token
    resp = s3.list_objects_v2(**kwargs)
    keys += [o["Key"] for o in resp.get("Contents", []) if o["Key"].endswith(".parquet")]
    if resp.get("IsTruncated"):
        token = resp["NextContinuationToken"]
    else:
        break

print(f"Found {len(keys)} parquet files")
frames = []
for key in keys:
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    # Extract state partition from key path
    table = pq.read_table(buf)
    df_part = table.to_pandas()
    for m in re.finditer(r"([^/=]+)=([^/]+)/", key):
        col, val = m.group(1), m.group(2)
        if col not in df_part.columns:
            df_part[col] = val
    frames.append(df_part)

ndvi = pd.concat(frames, ignore_index=True)
ndvi["date"] = pd.to_datetime(ndvi["date"])
print(f"Total rows: {len(ndvi):,}")
ndvi.head()

## 2. Basic Statistics

In [ ]:
print(f"Unique mandis: {ndvi['mandi_id'].nunique():,}")
print(f"Unique states: {ndvi['state'].nunique() if 'state' in ndvi.columns else 'N/A'}")
print(f"Date range: {ndvi['date'].min()} → {ndvi['date'].max()}")
print(f"NDVI range: {ndvi['ndvi'].min():.3f} → {ndvi['ndvi'].max():.3f}")
print(f"NDVI mean:  {ndvi['ndvi'].mean():.3f}")
print(f"NDVI std:   {ndvi['ndvi'].std():.3f}")
print()
ndvi.describe()

## 3. NDVI Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ndvi["ndvi"].dropna(), bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("NDVI")
axes[0].set_ylabel("Count")
axes[0].set_title("NDVI Distribution (all observations)")

# Box plot by month
ndvi["month"] = ndvi["date"].dt.month
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
month_data = [ndvi[ndvi["month"] == m]["ndvi"].dropna().values for m in range(1, 13)]
bp = axes[1].boxplot(month_data, labels=month_labels, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("lightgreen")
axes[1].set_ylabel("NDVI")
axes[1].set_title("NDVI by Month")

plt.tight_layout()
plt.show()

## 4. Coverage by State

In [ ]:
if "state" in ndvi.columns:
    state_counts = ndvi.groupby("state")["mandi_id"].nunique().sort_values(ascending=False)
    print(f"Mandis per state (top 15):")
    print(state_counts.head(15).to_string())
    
    fig, ax = plt.subplots(figsize=(12, 5))
    state_counts.head(20).plot(kind="bar", ax=ax, color="forestgreen", edgecolor="black")
    ax.set_ylabel("Number of unique mandis")
    ax.set_title("NDVI Coverage: Mandis per State (top 20)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No 'state' column — partition extraction may have failed")

## 5. Temporal Coverage per Mandi

In [ ]:
# How many months of data per mandi?
mandi_months = ndvi.groupby("mandi_id")["date"].apply(
    lambda x: x.dt.to_period("M").nunique()
)

print(f"Months per mandi — min: {mandi_months.min()}, max: {mandi_months.max()}, median: {mandi_months.median():.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(mandi_months, bins=range(1, mandi_months.max() + 2), edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Months of data per mandi")
axes[0].set_ylabel("Count of mandis")
axes[0].set_title("Temporal Coverage Distribution")

# Days of data per mandi (after forward-fill)
mandi_days = ndvi.groupby("mandi_id")["date"].nunique()
axes[1].hist(mandi_days, bins=30, edgecolor="black", alpha=0.7, color="steelblue")
axes[1].set_xlabel("Days of data per mandi")
axes[1].set_ylabel("Count of mandis")
axes[1].set_title("Daily Resolution Coverage")

plt.tight_layout()
plt.show()

## 6. Sample Mandi NDVI Time Series

In [ ]:
# Pick a few mandis with good coverage
top_mandis = mandi_days.sort_values(ascending=False).head(5).index.tolist()
print(f"Sample mandis: {top_mandis}")

fig, ax = plt.subplots(figsize=(14, 5))
for mandi in top_mandis:
    subset = ndvi[ndvi["mandi_id"] == mandi].sort_values("date")
    is_obs = subset.get("is_observed", pd.Series(dtype=int))
    ax.plot(subset["date"], subset["ndvi"], label=mandi[:40], alpha=0.8)
    # Mark actual observations
    if len(is_obs) > 0:
        obs = subset[is_obs == 1]
        ax.scatter(obs["date"], obs["ndvi"], marker="o", s=30, zorder=5)

ax.set_xlabel("Date")
ax.set_ylabel("NDVI")
ax.set_title("NDVI Time Series — Top 5 Mandis by Coverage")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

## 7. Spatial Distribution (lat/lon)

In [ ]:
if "latitude" in ndvi.columns and "longitude" in ndvi.columns:
    mandi_coords = ndvi.groupby("mandi_id").agg(
        lat=("latitude", "first"),
        lon=("longitude", "first"),
        mean_ndvi=("ndvi", "mean"),
    ).dropna()

    fig, ax = plt.subplots(figsize=(10, 10))
    sc = ax.scatter(
        mandi_coords["lon"], mandi_coords["lat"],
        c=mandi_coords["mean_ndvi"], cmap="RdYlGn", s=10, alpha=0.6,
        vmin=0.1, vmax=0.6,
    )
    plt.colorbar(sc, label="Mean NDVI")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Mandi Locations colored by Mean NDVI")
    ax.set_xlim(68, 98)
    ax.set_ylim(6, 38)
    plt.tight_layout()
    plt.show()
else:
    print("No lat/lon columns available")

## 8. Summary

**Key takeaways:**
- [ ] NDVI values are in the expected 0–0.7 range for Indian agriculture
- [ ] Forward-fill produces daily resolution from monthly observations
- [ ] Coverage varies by state — some states have more mandis than others
- [ ] Seasonal NDVI patterns should align with Kharif (Jul–Oct) and Rabi (Nov–Mar) crop cycles